In [ ]:
!unzip -q "/content/drive/MyDrive/BoneFractureYolo8.zip" -d "/content/dataset_cache"

In [ ]:
!pip install torchmetrics

In [ ]:
!pip install ultralytics

Архитектура №1

In [ ]:
from albumentations.pytorch import ToTensorV2
from torch import nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchmetrics.detection.iou import IntersectionOverUnion
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import albumentations as A
import cv2
import matplotlib.pyplot as plt
import os
import torch


class FractureDataset(Dataset):
    def __init__(self, images_dir, labels_dir, grid_size=7, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.grid_size = grid_size

        self.img_sorted = sorted(os.listdir(images_dir))

        self.transform_fracture = A.Compose([
            A.Resize(height=448, width=448),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()

        ], bbox_params=A.BboxParams(format='albumentations', label_fields=['class_labels']))
        self.transform_background = A.Compose([
            A.Resize(height=448, width=448),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.img_sorted)

    def __getitem__(self, idx):

        img_path = os.path.join(self.images_dir, self.img_sorted[idx])

        label_path = os.path.join(self.labels_dir, self.img_sorted[idx]).replace(".jpg", ".txt")

        labels = []
        boxes = []
        target = {}
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        with open(label_path, "r") as f:
            parts = f.read().split()
            curr_coords = []
            for item in parts:
                if "." not in item:
                    if len(curr_coords) >= 6:
                        labels.append(1)
                        x_coords = [round(float(curr_coords[i]), 5) for i in range(0, len(curr_coords), 2)]
                        y_coords = [round(float(curr_coords[i]), 5) for i in range(1, len(curr_coords), 2)]

                        x_max = max(0.0, min(1.0, max(x_coords)))
                        x_min = max(0.0, min(1.0, min(x_coords)))
                        y_max = max(0.0, min(1.0, max(y_coords)))
                        y_min = max(0.0, min(1.0, min(y_coords)))
                        boxes.append([x_min, y_min, x_max, y_max])
                    curr_coords = []
                else:
                    curr_coords.append(float(item))
            if len(curr_coords) >= 6:
                labels.append(1)

                x_coords = [round(float(curr_coords[i]), 5) for i in range(0, len(curr_coords), 2)]
                y_coords = [round(float(curr_coords[i]), 5) for i in range(1, len(curr_coords), 2)]

                x_max = max(0.0, min(1.0, max(x_coords)))
                x_min = max(0.0, min(1.0, min(x_coords)))
                y_max = max(0.0, min(1.0, max(y_coords)))
                y_min = max(0.0, min(1.0, min(y_coords)))
                boxes.append([x_min, y_min, x_max, y_max])

        if len(boxes) != 0:
            transformed = self.transform_fracture(image=image, bboxes=boxes, class_labels=labels)
            img_tensor = transformed['image']
            final_boxes = transformed['bboxes']
            target["labels"] = torch.as_tensor(labels, dtype=torch.int64)
            target["boxes"] = torch.as_tensor(boxes, dtype=torch.float32)
        else:
            transformed = self.transform_background(image=image)
            img_tensor = transformed['image']
            final_boxes = []
            target["labels"] = torch.zeros((0,), dtype=torch.int64)
            target["boxes"] = torch.zeros((0, 4), dtype=torch.float32)
        target["image_id"] = torch.tensor([idx], dtype=torch.int64)

        target_tensor = torch.zeros(self.grid_size, self.grid_size, 11)

        for box in final_boxes:
            x_min, y_min, x_max, y_max = box
            w_box = x_max - x_min
            h_box = y_max - y_min

            x_centered = (x_max + x_min) / 2
            y_centered = (y_max + y_min) / 2

            x_cell = int(x_centered * 7)
            y_cell = int(y_centered * 7)

            x_cell = max(0, min(6, x_cell))
            y_cell = max(0, min(6, y_cell))

            center_cell_x = (x_centered * self.grid_size) - x_cell
            center_cell_y = (y_centered * self.grid_size) - y_cell
            target_tensor[y_cell, x_cell] = torch.tensor([
                center_cell_x, center_cell_y, w_box, h_box, 1.0,
                center_cell_x, center_cell_y, w_box, h_box, 1.0,
                1.0
            ])

        return img_tensor, target_tensor


class ConvBlock1x1_3x3(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock1x1_3x3, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.1),

            nn.Conv2d(out_channels, in_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.LeakyReLU(0.1)

        )

    def forward(self, x):
        return self.conv(x)


class MyYOLOv1(nn.Module):
    def __init__(self, grid_size=7, num_anchors=2, num_classes=1):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 192, 3, 1, padding=1),
            nn.BatchNorm2d(192),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(192, 128, 1, 1, padding=0),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),

            nn.Conv2d(128, 256, 3, 1, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),

            nn.Conv2d(256, 256, 1, 1, padding=0),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),

            nn.Conv2d(256, 512, 3, 1, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),

            nn.MaxPool2d(2, 2),

            *[ConvBlock1x1_3x3(in_channels=512, out_channels=256) for _ in range(4)],

            nn.Conv2d(512, 512, 1, 1, padding=0),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),

            nn.Conv2d(512, 1024, 3, 1, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1),

            nn.MaxPool2d(2, 2),

            *[ConvBlock1x1_3x3(in_channels=1024, out_channels=512) for _ in range(2)],

            nn.Conv2d(1024, 1024, 3, 1, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1),

            nn.Conv2d(1024, 1024, 3, 2, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1),

            nn.Conv2d(1024, 1024, 3, 1, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1),

            nn.Conv2d(1024, 1024, 3, 1, padding=1),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.1)
        )
        self.conv_head = nn.Sequential(
            nn.Conv2d(1024, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, num_anchors * 5 + num_classes, kernel_size=1)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.conv_head(x)

        x = x.permute(0, 2, 3, 1).contiguous()
        return x


class YOLOLoss(nn.Module):
    def __init__(self, l_coord=5, l_no_obj=0.5, grid_size=7):
        super(YOLOLoss, self).__init__()
        self.l_coord = l_coord
        self.l_no_obj = l_no_obj
        self.grid_size = grid_size

        self.last_components = {}

    def forward(self, predictions, target):
        preds_grid = predictions
        frac_exist = target[..., 4]

        x1_local = torch.sigmoid(preds_grid[..., 0])
        y1_local = torch.sigmoid(preds_grid[..., 1])
        w1 = torch.sigmoid(preds_grid[..., 2])
        h1 = torch.sigmoid(preds_grid[..., 3])
        conf_1 = torch.sigmoid(preds_grid[..., 4])

        x2_local = torch.sigmoid(preds_grid[..., 5])
        y2_local = torch.sigmoid(preds_grid[..., 6])
        w2 = torch.sigmoid(preds_grid[..., 7])
        h2 = torch.sigmoid(preds_grid[..., 8])
        conf_2 = torch.sigmoid(preds_grid[..., 9])

        cell_x_ind = torch.arange(self.grid_size).repeat(self.grid_size, 1).unsqueeze(0).to(predictions.device)
        x1_global = (cell_x_ind + x1_local) / self.grid_size
        y1_global = (cell_x_ind.transpose(1, 2) + y1_local) / self.grid_size

        x1_min = x1_global - w1 / 2
        y1_min = y1_global - h1 / 2
        x1_max = x1_global + w1 / 2
        y1_max = y1_global + h1 / 2

        x2_global = (cell_x_ind + x2_local) / self.grid_size
        y2_global = (cell_x_ind.transpose(1, 2) + y2_local) / self.grid_size

        x2_min = x2_global - w2 / 2
        y2_min = y2_global - h2 / 2
        x2_max = x2_global + w2 / 2
        y2_max = y2_global + h2 / 2

        target_x_local = target[..., 0]
        target_y_local = target[..., 1]
        target_w = target[..., 2]
        target_h = target[..., 3]

        target_x_global = (cell_x_ind + target_x_local) / self.grid_size
        target_y_global = (cell_x_ind.transpose(1, 2) + target_y_local) / self.grid_size

        target_x_min = target_x_global - target_w / 2
        target_y_min = target_y_global - target_h / 2
        target_x_max = target_x_global + target_w / 2
        target_y_max = target_y_global + target_h / 2

        x1_min_inter = torch.max(target_x_min, x1_min)
        y1_min_inter = torch.max(target_y_min, y1_min)
        x1_max_inter = torch.min(target_x_max, x1_max)
        y1_max_inter = torch.min(target_y_max, y1_max)

        area_inter_1 = torch.clamp(x1_max_inter - x1_min_inter, min=0) * torch.clamp(y1_max_inter - y1_min_inter, min=0)
        area_box_1 = (x1_max - x1_min) * (y1_max - y1_min)
        area_target = (target_x_max - target_x_min) * (target_y_max - target_y_min)
        area_union_1 = area_box_1 + area_target - area_inter_1

        eps = 1e-6
        IoU1 = area_inter_1 / (area_union_1 + eps)

        x2_min_inter = torch.max(target_x_min, x2_min)
        y2_min_inter = torch.max(target_y_min, y2_min)
        x2_max_inter = torch.min(target_x_max, x2_max)
        y2_max_inter = torch.min(target_y_max, y2_max)

        area_inter_2 = torch.clamp(x2_max_inter - x2_min_inter, min=0) * torch.clamp(y2_max_inter - y2_min_inter, min=0)
        area_box_2 = (x2_max - x2_min) * (y2_max - y2_min)
        area_union_2 = area_box_2 + area_target - area_inter_2

        IoU2 = area_inter_2 / (area_union_2 + eps)

        max_iou, best_box_index = torch.max(torch.stack([IoU1, IoU2], dim=0), dim=0)

        best_x = best_box_index * x2_local + (1 - best_box_index) * x1_local
        best_y = best_box_index * y2_local + (1 - best_box_index) * y1_local

        w = best_box_index * w2 + (1 - best_box_index) * w1
        h = best_box_index * h2 + (1 - best_box_index) * h1
        best_w = torch.sqrt(torch.clamp(w, min=eps))
        best_h = torch.sqrt((torch.clamp(h, min=eps)))

        target_w_sqrt = torch.sqrt(target_w)
        target_h_sqrt = torch.sqrt(target_h)

        localization_loss = self.l_coord * torch.sum(
            frac_exist * (
                    (best_x - target_x_local) ** 2 +
                    (best_y - target_y_local) ** 2 +
                    (best_w - target_w_sqrt) ** 2 +
                    (best_h - target_h_sqrt) ** 2
            )
        )

        best_conf = best_box_index * conf_2 + (1 - best_box_index) * conf_1

        loss_obj = torch.sum(frac_exist * (best_conf - max_iou.detach()) ** 2) * 3.0

        obj_mask = frac_exist.bool()
        num_obj = obj_mask.sum().float()
        num_cells = torch.tensor(self.grid_size * self.grid_size * predictions.size(0),
                                 device=predictions.device).float()
        obj_ratio = num_obj / num_cells

        dynamic_no_obj_weight = self.l_no_obj * (1.0 - obj_ratio * 0.5)

        loss_no_obj = dynamic_no_obj_weight * torch.sum(
            (1 - frac_exist) * (conf_1 ** 2 + conf_2 ** 2)
        )

        pred_class = torch.sigmoid(preds_grid[..., 10])
        target_class = target[..., 10]

        loss_classification = torch.sum(frac_exist * (pred_class - target_class * 0.95) ** 2)

        total_loss = (localization_loss + loss_obj + loss_no_obj + loss_classification) / predictions.size(0)

        self.last_components = {
            'localization': localization_loss.item() / predictions.size(0),
            'obj': loss_obj.item() / predictions.size(0),
            'no_obj': loss_no_obj.item() / predictions.size(0),
            'class': loss_classification.item() / predictions.size(0)
        }

        return total_loss


def loss_graph(loss_history, is_val=False):
    plt.figure(figsize=(12, 8))
    plt.plot(loss_history, label="Train Loss")

    if is_val:
        plt.title("Valid Loss")
    else:
        plt.title("Train Loss")
    plt.xlabel("Эпоха", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.grid(True)
    plt.legend()

    plt.savefig("loss_graph.png")
    plt.show()


def validate(model, val_loader, criterion, device, map_metric):
    model.eval()
    total_loss = 0

    map_metric.reset()

    all_components = {'localization': 0, 'obj': 0, 'no_obj': 0, 'class': 0}

    with torch.no_grad():
        for (images, targets) in val_loader:
            images, targets = images.to(device), targets.to(device)
            predictions = model(images)
            loss = criterion(predictions, targets)
            total_loss += loss.item()

            m_preds, m_targets = format_for_metrics(predictions, targets, conf_threshold=0.05, grid_size=7)
            map_metric.update(m_preds, m_targets)

            comp = criterion.last_components
            for key in all_components:
                all_components[key] += comp[key]

    metrics_results = map_metric.compute()

    avg_loss = total_loss / len(val_loader)
    avg_components = {k: v / len(val_loader) for k, v in all_components.items()}
    return avg_loss, avg_components, metrics_results


def format_for_metrics(preds, targets, conf_threshold=0.5, img_size=448, grid_size=7):
    device = preds.device
    preds = preds.view(-1, grid_size, grid_size, 11)
    batch_size = preds.size(0)

    cell_x_ind = torch.arange(grid_size, device=device).repeat(grid_size, 1).unsqueeze(0)
    cell_y_ind = cell_x_ind.transpose(1, 2)

    metric_preds = []
    metric_targets = []

    for i in range(batch_size):
        img_preds = preds[i]
        pred_class = torch.sigmoid(img_preds[..., 10])

        x1 = torch.sigmoid(img_preds[..., 0])
        y1 = torch.sigmoid(img_preds[..., 1])
        w1 = torch.sigmoid(img_preds[..., 2])
        h1 = torch.sigmoid(img_preds[..., 3])
        conf_1 = torch.sigmoid(img_preds[..., 4])
        score_1 = conf_1 * pred_class
        mask_1 = score_1 > conf_threshold

        x2 = torch.sigmoid(img_preds[..., 5])
        y2 = torch.sigmoid(img_preds[..., 6])
        w2 = torch.sigmoid(img_preds[..., 7])
        h2 = torch.sigmoid(img_preds[..., 8])
        conf_2 = torch.sigmoid(img_preds[..., 9])
        score_2 = conf_2 * pred_class
        mask_2 = score_2 > conf_threshold

        boxes_list = []
        scores_list = []

        if mask_1.any():
            x_glob = (cell_x_ind[0][mask_1] + x1[mask_1]) / float(grid_size)
            y_glob = (cell_y_ind[0][mask_1] + y1[mask_1]) / float(grid_size)
            x_min = (x_glob - w1[mask_1] / 2.0) * img_size
            y_min = (y_glob - h1[mask_1] / 2.0) * img_size
            x_max = (x_glob + w1[mask_1] / 2.0) * img_size
            y_max = (y_glob + h1[mask_1] / 2.0) * img_size

            b1 = torch.stack([x_min, y_min, x_max, y_max], dim=1)
            boxes_list.append(b1)
            scores_list.append(score_1[mask_1])

        if mask_2.any():
            x_glob = (cell_x_ind[0][mask_2] + x2[mask_2]) / float(grid_size)
            y_glob = (cell_y_ind[0][mask_2] + y2[mask_2]) / float(grid_size)
            x_min = (x_glob - w2[mask_2] / 2.0) * img_size
            y_min = (y_glob - h2[mask_2] / 2.0) * img_size
            x_max = (x_glob + w2[mask_2] / 2.0) * img_size
            y_max = (y_glob + h2[mask_2] / 2.0) * img_size

            b2 = torch.stack([x_min, y_min, x_max, y_max], dim=1)
            boxes_list.append(b2)
            scores_list.append(score_2[mask_2])

        if len(boxes_list) > 0:
            final_boxes = torch.cat(boxes_list, dim=0)
            final_scores = torch.cat(scores_list, dim=0)
            final_labels = torch.zeros(final_boxes.size(0), dtype=torch.int64, device=device)

            metric_preds.append({
                "boxes": final_boxes,
                "scores": final_scores,
                "labels": final_labels
            })
        else:
            metric_preds.append({
                "boxes": torch.zeros((0, 4), dtype=torch.float32, device=device),
                "scores": torch.zeros((0,), dtype=torch.float32, device=device),
                "labels": torch.zeros((0,), dtype=torch.int64, device=device)
            })

        img_target = targets[i]
        target_mask = img_target[..., 4] > 0.5

        if target_mask.any():
            tx = img_target[..., 0][target_mask]
            ty = img_target[..., 1][target_mask]
            tw = img_target[..., 2][target_mask]
            th = img_target[..., 3][target_mask]

            tx_glob = (cell_x_ind[0][target_mask] + tx) / float(grid_size)
            ty_glob = (cell_y_ind[0][target_mask] + ty) / float(grid_size)
            tx_min = (tx_glob - tw / 2.0) * img_size
            ty_min = (ty_glob - th / 2.0) * img_size
            tx_max = (tx_glob + tw / 2.0) * img_size
            ty_max = (ty_glob + th / 2.0) * img_size

            final_t_boxes = torch.stack([tx_min, ty_min, tx_max, ty_max], dim=1)
            final_t_labels = torch.zeros(final_t_boxes.size(0), dtype=torch.int64, device=device)

            metric_targets.append({
                "boxes": final_t_boxes,
                "labels": final_t_labels
            })
        else:
            metric_targets.append({
                "boxes": torch.zeros((0, 4), dtype=torch.float32, device=device),
                "labels": torch.zeros((0,), dtype=torch.int64, device=device)
            })

    return metric_preds, metric_targets


def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / (union + 1e-6)


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    IMAGES_DIR = "/content/dataset_cache/train/images"
    LABELS_DIR = "/content/dataset_cache/train/labels"

    VAL_IMAGES_DIR = "/content/dataset_cache/valid/images"
    VAL_LABELS_DIR = "/content/dataset_cache/valid/labels"

    val_dataset = FractureDataset(VAL_IMAGES_DIR, VAL_LABELS_DIR, grid_size=7)
    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        drop_last=False
    )

    train_dataset = FractureDataset(IMAGES_DIR, LABELS_DIR)
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=32,
        shuffle=True,
        num_workers=2,
        drop_last=True
    )
    model = MyYOLOv1().to(device)
    criterion = YOLOLoss(l_coord=5, l_no_obj=0.05, grid_size=7)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

    map_metric = MeanAveragePrecision(box_format="xyxy", class_metrics=True).to(device)

    NUM_EPOCHS = 50

    loss_history = []
    val_history = []

    for epoch in range(0, NUM_EPOCHS):
        model.train()
        epoch_loss = 0

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        for batch_idx, (images, targets) in enumerate(train_loader):

            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            predictions = model(images)

            loss = criterion(predictions, targets)

            if (batch_idx + 1) % 10 == 0:
                print(
                    f"Эпоха [{epoch + 1}/{NUM_EPOCHS}], Шаг [{batch_idx + 1}/{len(train_loader)}], Loss: {loss.item():.3f}")
                comp = criterion.last_components

                print(
                    f"Train  Loc: {comp['localization']:.3f}, Obj: {comp['obj']:.3f}, NoObj: {comp['no_obj']:.3f}, Class: {comp['class']:.3f}")
                print()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)
        loss_history.append(epoch_loss)
        print()
        print(f"Эпоха [{epoch + 1}/{NUM_EPOCHS}], Средний Loss: {epoch_loss:.3f}")
        val_loss, val_components, val_metrics = validate(model, val_loader, criterion, device, map_metric)
        print(
            f"Val  Loc: {val_components['localization']:.3f}, Obj: {val_components['obj']:.3f}, NoObj: {val_components['no_obj']:.3f}, Class: {val_components['class']:.3f}")
        val_history.append(val_loss)

        if 'map_per_class' in val_metrics and val_metrics['map_per_class'].numel() > 0:
            precision = val_metrics['map_per_class'].item() if val_metrics['map_per_class'].dim() == 0 else \
                val_metrics['map_per_class'][0].item()
        else:
            precision = 0.0

        if 'mar_100_per_class' in val_metrics and val_metrics['mar_100_per_class'].numel() > 0:
            recall = val_metrics['mar_100_per_class'].item() if val_metrics['mar_100_per_class'].dim() == 0 else \
                val_metrics['mar_100_per_class'][0].item()
        else:
            recall = 0.0
        f1_score = 2 * (precision * recall) / (precision + recall + 1e-6)

        print(f"\n  Метрики на валидации:")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall:    {recall:.4f}")
        print(f"  F1-Score:  {f1_score:.4f}")
        print(f"  mAP_50: {val_metrics['map_50'].item():.4f}")

        print("-----------------------------------------------------------------")

        if (epoch + 1) % 10 == 0:
            checkpoint_path = f"yolov1_fracture_epoch_{epoch + 1}.pth"
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Веса сохранены в {checkpoint_path}")
        scheduler.step()
    loss_graph(loss_history)
    loss_graph(val_history, True)


In [ ]:
from albumentations.pytorch import ToTensorV2
from torch import nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchmetrics.detection.iou import IntersectionOverUnion
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import albumentations as A
import cv2
import matplotlib.pyplot as plt
import os
import torch
import torchvision.models as models


class FractureDataset(Dataset):
    def __init__(self, images_dir, labels_dir, grid_size=7, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.grid_size = grid_size

        self.img_sorted = sorted(os.listdir(images_dir))

        self.transform_fracture = A.Compose([
            A.Resize(height=448, width=448),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()

        ], bbox_params=A.BboxParams(format='albumentations', label_fields=['class_labels']))
        self.transform_background = A.Compose([
            A.Resize(height=448, width=448),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.img_sorted)

    def __getitem__(self, idx):

        img_path = os.path.join(self.images_dir, self.img_sorted[idx])

        label_path = os.path.join(self.labels_dir, self.img_sorted[idx]).replace(".jpg", ".txt")

        labels = []
        boxes = []
        target = {}
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        with open(label_path, "r") as f:
            parts = f.read().split()
            curr_coords = []
            for item in parts:
                if "." not in item:
                    if len(curr_coords) >= 6:
                        labels.append(1)
                        x_coords = [round(float(curr_coords[i]), 5) for i in range(0, len(curr_coords), 2)]
                        y_coords = [round(float(curr_coords[i]), 5) for i in range(1, len(curr_coords), 2)]

                        x_max = max(0.0, min(1.0, max(x_coords)))
                        x_min = max(0.0, min(1.0, min(x_coords)))
                        y_max = max(0.0, min(1.0, max(y_coords)))
                        y_min = max(0.0, min(1.0, min(y_coords)))
                        boxes.append([x_min, y_min, x_max, y_max])
                    curr_coords = []
                else:
                    curr_coords.append(float(item))
            if len(curr_coords) >= 6:
                labels.append(1)

                x_coords = [round(float(curr_coords[i]), 5) for i in range(0, len(curr_coords), 2)]
                y_coords = [round(float(curr_coords[i]), 5) for i in range(1, len(curr_coords), 2)]

                x_max = max(0.0, min(1.0, max(x_coords)))
                x_min = max(0.0, min(1.0, min(x_coords)))
                y_max = max(0.0, min(1.0, max(y_coords)))
                y_min = max(0.0, min(1.0, min(y_coords)))
                boxes.append([x_min, y_min, x_max, y_max])

        if len(boxes) != 0:
            transformed = self.transform_fracture(image=image, bboxes=boxes, class_labels=labels)
            img_tensor = transformed['image']
            final_boxes = transformed['bboxes']
            target["labels"] = torch.as_tensor(labels, dtype=torch.int64)
            target["boxes"] = torch.as_tensor(boxes, dtype=torch.float32)
        else:
            transformed = self.transform_background(image=image)
            img_tensor = transformed['image']
            final_boxes = []
            target["labels"] = torch.zeros((0,), dtype=torch.int64)
            target["boxes"] = torch.zeros((0, 4), dtype=torch.float32)
        target["image_id"] = torch.tensor([idx], dtype=torch.int64)

        target_tensor = torch.zeros(self.grid_size, self.grid_size, 11)

        for box in final_boxes:
            x_min, y_min, x_max, y_max = box
            w_box = x_max - x_min
            h_box = y_max - y_min

            x_centered = (x_max + x_min) / 2
            y_centered = (y_max + y_min) / 2

            x_cell = int(x_centered * self.grid_size)
            y_cell = int(y_centered * self.grid_size)

            x_cell = max(0, min(13, x_cell))
            y_cell = max(0, min(13, y_cell))

            center_cell_x = (x_centered * self.grid_size) - x_cell
            center_cell_y = (y_centered * self.grid_size) - y_cell
            target_tensor[y_cell, x_cell] = torch.tensor([
                center_cell_x, center_cell_y, w_box, h_box, 1.0,
                center_cell_x, center_cell_y, w_box, h_box, 1.0,
                1.0
            ])

        return img_tensor, target_tensor


class FPNYOLOv1(nn.Module):
    def __init__(self, grid_size=7, num_anchors=2, num_classes=1):
        super().__init__()

        resnet = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)

        self.backbone = nn.Sequential(*list(resnet.children())[:-2])

        self.fpn_1 = nn.Conv2d(512, 256, kernel_size=1)
        self.fpn_output = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1)
        )

        self.downsample = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
        )

        out_channels = num_anchors * 5 + num_classes
        self.output_layer = nn.Conv2d(256, out_channels, kernel_size=1)

    def forward(self, x):
        x = self.backbone(x)

        lateral = self.fpn_1(x)

        x = self.fpn_output(lateral)

        x = self.downsample(x)

        x = self.output_layer(x)
        x = x.permute(0, 2, 3, 1).contiguous()

        return x


class YOLOLoss(nn.Module):
    def __init__(self, l_coord=5, l_no_obj=0.5, grid_size=7):
        super(YOLOLoss, self).__init__()
        self.l_coord = l_coord
        self.l_no_obj = l_no_obj
        self.grid_size = grid_size

        self.last_components = {}

    def forward(self, predictions, target):
        preds_grid = predictions
        frac_exist = target[..., 4]

        x1_local = torch.sigmoid(preds_grid[..., 0])
        y1_local = torch.sigmoid(preds_grid[..., 1])
        w1 = torch.sigmoid(preds_grid[..., 2])
        h1 = torch.sigmoid(preds_grid[..., 3])
        conf_1 = torch.sigmoid(preds_grid[..., 4])

        x2_local = torch.sigmoid(preds_grid[..., 5])
        y2_local = torch.sigmoid(preds_grid[..., 6])
        w2 = torch.sigmoid(preds_grid[..., 7])
        h2 = torch.sigmoid(preds_grid[..., 8])
        conf_2 = torch.sigmoid(preds_grid[..., 9])

        cell_x_ind = torch.arange(self.grid_size).repeat(self.grid_size, 1).unsqueeze(0).to(predictions.device)
        x1_global = (cell_x_ind + x1_local) / self.grid_size
        y1_global = (cell_x_ind.transpose(1, 2) + y1_local) / self.grid_size

        x1_min = x1_global - w1 / 2
        y1_min = y1_global - h1 / 2
        x1_max = x1_global + w1 / 2
        y1_max = y1_global + h1 / 2

        x2_global = (cell_x_ind + x2_local) / self.grid_size
        y2_global = (cell_x_ind.transpose(1, 2) + y2_local) / self.grid_size

        x2_min = x2_global - w2 / 2
        y2_min = y2_global - h2 / 2
        x2_max = x2_global + w2 / 2
        y2_max = y2_global + h2 / 2

        target_x_local = target[..., 0]
        target_y_local = target[..., 1]
        target_w = target[..., 2]
        target_h = target[..., 3]

        target_x_global = (cell_x_ind + target_x_local) / self.grid_size
        target_y_global = (cell_x_ind.transpose(1, 2) + target_y_local) / self.grid_size

        target_x_min = target_x_global - target_w / 2
        target_y_min = target_y_global - target_h / 2
        target_x_max = target_x_global + target_w / 2
        target_y_max = target_y_global + target_h / 2

        x1_min_inter = torch.max(target_x_min, x1_min)
        y1_min_inter = torch.max(target_y_min, y1_min)
        x1_max_inter = torch.min(target_x_max, x1_max)
        y1_max_inter = torch.min(target_y_max, y1_max)

        area_inter_1 = torch.clamp(x1_max_inter - x1_min_inter, min=0) * torch.clamp(y1_max_inter - y1_min_inter, min=0)
        area_box_1 = (x1_max - x1_min) * (y1_max - y1_min)
        area_target = (target_x_max - target_x_min) * (target_y_max - target_y_min)
        area_union_1 = area_box_1 + area_target - area_inter_1

        eps = 1e-6
        IoU1 = area_inter_1 / (area_union_1 + eps)

        x2_min_inter = torch.max(target_x_min, x2_min)
        y2_min_inter = torch.max(target_y_min, y2_min)
        x2_max_inter = torch.min(target_x_max, x2_max)
        y2_max_inter = torch.min(target_y_max, y2_max)

        area_inter_2 = torch.clamp(x2_max_inter - x2_min_inter, min=0) * torch.clamp(y2_max_inter - y2_min_inter, min=0)
        area_box_2 = (x2_max - x2_min) * (y2_max - y2_min)
        area_union_2 = area_box_2 + area_target - area_inter_2

        IoU2 = area_inter_2 / (area_union_2 + eps)

        max_iou, best_box_index = torch.max(torch.stack([IoU1, IoU2], dim=0), dim=0)

        best_x = best_box_index * x2_local + (1 - best_box_index) * x1_local
        best_y = best_box_index * y2_local + (1 - best_box_index) * y1_local

        w = best_box_index * w2 + (1 - best_box_index) * w1
        h = best_box_index * h2 + (1 - best_box_index) * h1
        best_w = torch.sqrt(torch.clamp(w, min=eps))
        best_h = torch.sqrt((torch.clamp(h, min=eps)))

        target_w_sqrt = torch.sqrt(target_w)
        target_h_sqrt = torch.sqrt(target_h)

        localization_loss = self.l_coord * torch.sum(
            frac_exist * (
                    (best_x - target_x_local) ** 2 +
                    (best_y - target_y_local) ** 2 +
                    (best_w - target_w_sqrt) ** 2 +
                    (best_h - target_h_sqrt) ** 2
            )
        )

        best_conf = best_box_index * conf_2 + (1 - best_box_index) * conf_1

        loss_obj = torch.sum(frac_exist * (best_conf - max_iou.detach()) ** 2) * 3.0

        obj_mask = frac_exist.bool()
        num_obj = obj_mask.sum().float()
        num_cells = torch.tensor(self.grid_size * self.grid_size * predictions.size(0), device=predictions.device).float()
        obj_ratio = num_obj / num_cells

        dynamic_no_obj_weight = self.l_no_obj * (1.0 - obj_ratio * 0.5)

        loss_no_obj = dynamic_no_obj_weight * torch.sum(
            (1 - frac_exist) * (conf_1 ** 2 + conf_2 ** 2)
        )

        pred_class = torch.sigmoid(preds_grid[..., 10])
        target_class = target[..., 10]

        loss_classification = torch.sum(frac_exist * (pred_class - target_class * 0.95) ** 2)

        total_loss = (localization_loss + loss_obj + loss_no_obj + loss_classification) / predictions.size(0)

        self.last_components = {
            'localization': localization_loss.item() / predictions.size(0),
            'obj': loss_obj.item() / predictions.size(0),
            'no_obj': loss_no_obj.item() / predictions.size(0),
            'class': loss_classification.item() / predictions.size(0)
        }

        return total_loss


def loss_graph(loss_history, is_val=False):
    plt.figure(figsize=(12, 8))
    plt.plot(loss_history, label="Train Loss")

    if is_val:
        plt.title("Valid Loss")
    else:
        plt.title("Train Loss")
    plt.xlabel("Эпоха", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.grid(True)
    plt.legend()

    plt.savefig("loss_graph.png")
    plt.show()


def validate(model, val_loader, criterion, device, map_metric):
    model.eval()
    total_loss = 0

    map_metric.reset()

    all_components = {'localization': 0, 'obj': 0, 'no_obj': 0, 'class': 0}

    with torch.no_grad():
        for (images, targets) in val_loader:
            images, targets = images.to(device), targets.to(device)
            predictions = model(images)
            loss = criterion(predictions, targets)
            total_loss += loss.item()

            m_preds, m_targets = format_for_metrics(predictions, targets, conf_threshold=0.05, grid_size=7)
            map_metric.update(m_preds, m_targets)

            comp = criterion.last_components
            for key in all_components:
                all_components[key] += comp[key]

    metrics_results = map_metric.compute()

    avg_loss = total_loss / len(val_loader)
    avg_components = {k: v / len(val_loader) for k, v in all_components.items()}
    return avg_loss, avg_components, metrics_results


def format_for_metrics(preds, targets, conf_threshold=0.5, img_size=448, grid_size=7):
    device = preds.device
    preds = preds.view(-1, grid_size, grid_size, 11)
    batch_size = preds.size(0)

    cell_x_ind = torch.arange(grid_size, device=device).repeat(grid_size, 1).unsqueeze(0)
    cell_y_ind = cell_x_ind.transpose(1, 2)

    metric_preds = []

    metric_targets = []

    for i in range(batch_size):
        img_preds = preds[i]
        pred_class = torch.sigmoid(img_preds[..., 10])

        x1 = torch.sigmoid(img_preds[..., 0])
        y1 = torch.sigmoid(img_preds[..., 1])
        w1 = torch.exp(torch.clamp(img_preds[..., 2], max=2.0))
        h1 = torch.exp(torch.clamp(img_preds[..., 3], max=2.0))
        conf_1 = torch.sigmoid(img_preds[..., 4])
        score_1 = conf_1 * pred_class
        mask_1 = score_1 > conf_threshold

        x2 = torch.sigmoid(img_preds[..., 5])
        y2 = torch.sigmoid(img_preds[..., 6])
        w2 = torch.exp(torch.clamp(img_preds[..., 7], max=2.0))
        h2 = torch.exp(torch.clamp(img_preds[..., 8], max=2.0))
        conf_2 = torch.sigmoid(img_preds[..., 9])
        score_2 = conf_2 * pred_class
        mask_2 = score_2 > conf_threshold

        boxes_list = []
        scores_list = []

        if mask_1.any():
            x_glob = (cell_x_ind[0][mask_1] + x1[mask_1]) / float(grid_size)
            y_glob = (cell_y_ind[0][mask_1] + y1[mask_1]) / float(grid_size)
            x_min = (x_glob - w1[mask_1] / 2.0) * img_size
            y_min = (y_glob - h1[mask_1] / 2.0) * img_size
            x_max = (x_glob + w1[mask_1] / 2.0) * img_size
            y_max = (y_glob + h1[mask_1] / 2.0) * img_size

            b1 = torch.stack([x_min, y_min, x_max, y_max], dim=1)
            boxes_list.append(b1)
            scores_list.append(score_1[mask_1])

        if mask_2.any():
            x_glob = (cell_x_ind[0][mask_2] + x2[mask_2]) / float(grid_size)
            y_glob = (cell_y_ind[0][mask_2] + y2[mask_2]) / float(grid_size)
            x_min = (x_glob - w2[mask_2] / 2.0) * img_size
            y_min = (y_glob - h2[mask_2] / 2.0) * img_size
            x_max = (x_glob + w2[mask_2] / 2.0) * img_size
            y_max = (y_glob + h2[mask_2] / 2.0) * img_size

            b2 = torch.stack([x_min, y_min, x_max, y_max], dim=1)
            boxes_list.append(b2)
            scores_list.append(score_2[mask_2])

        if len(boxes_list) > 0:
            final_boxes = torch.cat(boxes_list, dim=0)
            final_scores = torch.cat(scores_list, dim=0)
            final_labels = torch.zeros(final_boxes.size(0), dtype=torch.int64, device=device)

            metric_preds.append({
                "boxes": final_boxes,
                "scores": final_scores,
                "labels": final_labels
            })
        else:
            metric_preds.append({
                "boxes": torch.zeros((0, 4), dtype=torch.float32, device=device),
                "scores": torch.zeros((0,), dtype=torch.float32, device=device),
                "labels": torch.zeros((0,), dtype=torch.int64, device=device)
            })

        img_target = targets[i]
        target_mask = img_target[..., 4] > 0.5

        if target_mask.any():
            tx = img_target[..., 0][target_mask]
            ty = img_target[..., 1][target_mask]
            tw = img_target[..., 2][target_mask]
            th = img_target[..., 3][target_mask]

            tx_glob = (cell_x_ind[0][target_mask] + tx) / float(grid_size)
            ty_glob = (cell_y_ind[0][target_mask] + ty) / float(grid_size)
            tx_min = (tx_glob - tw / 2.0) * img_size
            ty_min = (ty_glob - th / 2.0) * img_size
            tx_max = (tx_glob + tw / 2.0) * img_size
            ty_max = (ty_glob + th / 2.0) * img_size

            final_t_boxes = torch.stack([tx_min, ty_min, tx_max, ty_max], dim=1)
            final_t_labels = torch.zeros(final_t_boxes.size(0), dtype=torch.int64, device=device)

            metric_targets.append({
                "boxes": final_t_boxes,
                "labels": final_t_labels
            })
        else:
            metric_targets.append({
                "boxes": torch.zeros((0, 4), dtype=torch.float32, device=device),
                "labels": torch.zeros((0,), dtype=torch.int64, device=device)
            })

    return metric_preds, metric_targets


def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / (union + 1e-6)


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    IMAGES_DIR = "/content/dataset_cache/train/images"
    LABELS_DIR = "/content/dataset_cache/train/labels"

    VAL_IMAGES_DIR = "/content/dataset_cache/valid/images"
    VAL_LABELS_DIR = "/content/dataset_cache/valid/labels"

    val_dataset = FractureDataset(VAL_IMAGES_DIR, VAL_LABELS_DIR, grid_size=7)
    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        drop_last=False
    )

    train_dataset = FractureDataset(IMAGES_DIR, LABELS_DIR)
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=32,
        shuffle=True,
        num_workers=2,
        drop_last=True
    )
    model = FPNYOLOv1().to(device)
    criterion = YOLOLoss(l_coord=5, l_no_obj=0.05, grid_size=7)

    optimizer = torch.optim.Adam(model.parameters(), weight_decay=1e-4, lr=0.0003)
    scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

    map_metric = MeanAveragePrecision(box_format="xyxy", class_metrics=True).to(device)

    NUM_EPOCHS = 50

    loss_history = []
    val_history = []

    for epoch in range(0, NUM_EPOCHS):
        model.train()
        epoch_loss = 0

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        for batch_idx, (images, targets) in enumerate(train_loader):

            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            predictions = model(images)

            loss = criterion(predictions, targets)

            if (batch_idx + 1) % 10 == 0:
                print(
                    f"Эпоха [{epoch + 1}/{NUM_EPOCHS}], Шаг [{batch_idx + 1}/{len(train_loader)}], Train Loss: {loss.item():.3f}")
                comp = criterion.last_components
                print(
                    f"Train  Loc: {comp['localization']:.3f}, Obj: {comp['obj']:.3f}, NoObj: {comp['no_obj']:.3f}, Class: {comp['class']:.3f}")
                print()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)
        loss_history.append(epoch_loss)
        print()
        print(f"Эпоха [{epoch + 1}/{NUM_EPOCHS}], Средний Loss: {epoch_loss:.3f}")
        val_loss, val_components, val_metrics = validate(model, val_loader, criterion, device, map_metric)
        print(f"Validation Loss: {val_loss:.3f}")
        print(
            f"Val  Loc: {val_components['localization']:.3f}, Obj: {val_components['obj']:.3f}, NoObj: {val_components['no_obj']:.3f}, Class: {val_components['class']:.3f}")
        val_history.append(val_loss)

        if 'map_per_class' in val_metrics and val_metrics['map_per_class'].numel() > 0:
            precision = val_metrics['map_per_class'].item() if val_metrics['map_per_class'].dim() == 0 else \
                val_metrics['map_per_class'][0].item()
        else:
            precision = 0.0

        if 'mar_100_per_class' in val_metrics and val_metrics['mar_100_per_class'].numel() > 0:
            recall = val_metrics['mar_100_per_class'].item() if val_metrics['mar_100_per_class'].dim() == 0 else \
                val_metrics['mar_100_per_class'][0].item()
        else:
            recall = 0.0
        f1_score = 2 * (precision * recall) / (precision + recall + 1e-6)

        print(f"  Метрики на валидации (conf > 0.05, IoU = 0.5):")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall:    {recall:.4f}")
        print(f"  F1-Score:  {f1_score:.4f}")
        print(f"  mAP_50: {val_metrics['map_50'].item():.4f}")

        print("-----------------------------------------------------------------")

        if (epoch + 1) % 10 == 0:
            checkpoint_path = f"yolov1_fracture_epoch_{epoch + 1}.pth"
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Веса сохранены в {checkpoint_path}")
        scheduler.step()
    loss_graph(loss_history)
    loss_graph(val_history, True)


Архитектура №2

In [ ]:
import os
import yaml
import torch
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO


def fix_labels():
    splits = ['train', 'valid']

    for split in splits:
        labels_dir = f'/content/dataset_cache/{split}/labels'

        files_processed = 0
        files_fixed = 0

        for label_file in os.listdir(labels_dir):
            if not label_file.endswith('.txt'):
                continue

            file_path = os.path.join(labels_dir, label_file)
            files_processed += 1

            with open(file_path, 'r') as f:
                lines = f.readlines()

            needs_fix = False
            fixed_lines = []

            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    original_class = int(parts[0])
                    if original_class != 0:
                        needs_fix = True
                        parts[0] = '0'
                    fixed_lines.append(' '.join(parts))

            if needs_fix:
                files_fixed += 1
                with open(file_path, 'w') as f:
                    f.write('\n'.join(fixed_lines))

        cache_file = f'/content/dataset_cache/{split}/labels.cache'
        if os.path.exists(cache_file):
            os.remove(cache_file)


def loss_graph(loss_history, is_val=False):
    plt.figure(figsize=(12, 8))

    if is_val:
        plt.plot(loss_history, label="Valid Loss")
        plt.title("Valid Loss")
    else:
        plt.plot(loss_history, label="Train Loss")
        plt.title("Train Loss")
    plt.xlabel("Эпоха", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.grid(True)
    plt.legend()

    plt.savefig("loss_graph.png")
    plt.show()


def create_data_yaml():
    data = {
        'path': '/content/dataset_cache',
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'valid/images',
        'nc': 1,
        'names': {
            0: 'fracture'
        }
    }

    with open('data.yaml', 'w') as f:
        yaml.dump(data, f, default_flow_style=False, allow_unicode=True)

    # print(yaml.dump(data, default_flow_style=False))

    return 'data.yaml'


def train_yolov8(data_yaml_path, model_size='m', epochs=50, batch=16, imgsz=640):
    model = YOLO(f'yolov8{model_size}.pt')

    results = model.train(
        data=data_yaml_path,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        workers=2,
        patience=30,
        save=True,
        save_period=10,
        project='yolov8_fracture',
        name=f'yolov8{model_size}_fracture_final',
        exist_ok=True,
        flipud=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.3,
        copy_paste=0.5,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.5,
        scale=0.8
    )

    return model, results


def evaluate_model(model, data_yaml_path):
    metrics = model.val(data=data_yaml_path)

    print(f"\n Метрики на валидации:")
    print(f"  mAP50:     {metrics.box.map50:.4f}")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall:    {metrics.box.mr:.4f}")

    precision = metrics.box.mp
    recall = metrics.box.mr
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
    print(f"  F1-Score:  {f1:.4f}")

    return metrics


if __name__ == "__main__":
    fix_labels()

    data_yaml = create_data_yaml()

    model, results = train_yolov8(
        data_yaml_path=data_yaml,
        model_size='m',
        epochs=90,
        batch=8,
        imgsz=640
    )

    metrics = evaluate_model(model, data_yaml)

    results_dir = f'runs/detect/yolov8_fracture/yolov8m_fracture_final'
    results_file = os.path.join(results_dir, 'results.csv')

    loss_history = []
    val_loss_history = []

    if os.path.exists(results_file):
        df = pd.read_csv(results_file)

        if 'train/box_loss' in df.columns and 'train/cls_loss' in df.columns and 'train/dfl_loss' in df.columns:
            loss_history = (df['train/box_loss'] + df['train/cls_loss'] + df['train/dfl_loss']).values.tolist()
        else:
            print(1)

        if 'val/box_loss' in df.columns and 'val/cls_loss' in df.columns and 'val/dfl_loss' in df.columns:
            val_loss_history = (df['val/box_loss'] + df['val/cls_loss'] + df['val/dfl_loss']).values.tolist()
        else:
            print(2)

        if loss_history:
            loss_graph(loss_history, is_val=False)

        if val_loss_history:
            loss_graph(val_loss_history, is_val=True)

    else:
        print(f"Файл не найден")

    model.export(format='onnx')
    torch.save(model.state_dict(), 'yolov8_fracture_best.pth')

    precision = metrics.box.mp
    recall = metrics.box.mr
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)

    print(f"\n метрики:")
    print(f"  mAP50:     {metrics.box.map50:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
